In [ ]:
# Compare rho FFT mode amplitudes for diff_0 vs diff_10.
# Creates 15 subplots (k=1..15), each with both datasets overlaid.

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASETS = [
    ("diff_0", Path("diff_0")),
    ("diff_10", Path("diff_10")),
    # ("diff_100", Path("diff_100")),
    # ("diff_1000", Path("diff_1000")),
]
PATTERN = "rho_field_samples_*x*_step_*.csv"
K_MAX = 15
DT = None  # set to simulation dt to plot physical time
EPS = 1e-30


def extract_step(path: Path):
    m = re.search(r"step_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else -1


def compute_modes(folder: Path, k_max: int):
    files = sorted(folder.glob(PATTERN), key=extract_step)
    if not files:
        raise FileNotFoundError(f"No files in {folder} matching {PATTERN}")

    steps = []
    mode_amp = []

    for f in files:
        df = pd.read_csv(f, usecols=["i", "j", "step", "rho"])

        nx = int(df["i"].max()) + 1
        ny = int(df["j"].max()) + 1
        if len(df) != nx * ny:
            raise ValueError(f"{f.name}: expected {nx*ny} rows, got {len(df)}")

        df = df.sort_values(["j", "i"], kind="mergesort")
        rho = df["rho"].to_numpy().reshape(ny, nx)

        profile_x = rho.mean(axis=0)
        fft_vals = np.fft.rfft(profile_x)

        this_kmax = min(k_max, fft_vals.size - 1)
        amps = np.zeros(k_max, dtype=float)
        if this_kmax > 0:
            amps[:this_kmax] = np.abs(fft_vals[1:this_kmax + 1]) / profile_x.size

        steps.append(int(df["step"].iloc[0]))
        mode_amp.append(amps)

    return np.asarray(steps), np.asarray(mode_amp)


results = {}
for label, folder in DATASETS:
    steps, amps = compute_modes(folder, K_MAX)
    t = steps if DT is None else steps * float(DT)
    results[label] = {"steps": steps, "amps": amps, "t": t}

t_label = "step" if DT is None else "time"

fig, axes = plt.subplots(5, 3, figsize=(15, 16), sharex=True)
axes = axes.ravel()

for k in range(1, K_MAX + 1):
    ax = axes[k - 1]
    for label, _folder in DATASETS:
        y = np.log10(np.maximum(results[label]["amps"][:, k - 1], EPS))
        ax.plot(results[label]["t"], y, lw=1.6, label=label)
    ax.set_title(f"k={k}")
    ax.grid(True, alpha=0.3)

for ax in axes[-3:]:
    ax.set_xlabel(t_label)
for i in [0, 3, 6, 9, 12]:
    axes[i].set_ylabel(r"$\log_{10}|\rho_k|$")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=len(DATASETS), frameon=False)
fig.suptitle("Mode comparison (rho)", y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

for label, _folder in DATASETS:
    steps = results[label]["steps"]
    print(f"{label}: {len(steps)} files, step range [{steps.min()}, {steps.max()}]")

In [ ]:
# Compare phi FFT mode amplitudes for diff_0 vs diff_10.
# Creates 15 subplots (k=1..15), each with both datasets overlaid.

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASETS = [
    ("diff_0", Path("diff_0")),
    ("diff_10", Path("diff_10")),
    # ("diff_100", Path("diff_100")),
    # ("diff_1000", Path("diff_1000")),
]
PATTERN = "phi_field_samples_*x*_step_*.csv"
K_MAX = 15
DT = None  # set to simulation dt to plot physical time
EPS = 1e-30


def extract_step(path: Path):
    m = re.search(r"step_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else -1


def compute_modes(folder: Path, k_max: int):
    files = sorted(folder.glob(PATTERN), key=extract_step)
    if not files:
        raise FileNotFoundError(f"No files in {folder} matching {PATTERN}")

    steps = []
    mode_amp = []

    for f in files:
        df = pd.read_csv(f, usecols=["i", "j", "step", "phi"])

        nx = int(df["i"].max()) + 1
        ny = int(df["j"].max()) + 1
        if len(df) != nx * ny:
            raise ValueError(f"{f.name}: expected {nx*ny} rows, got {len(df)}")

        df = df.sort_values(["j", "i"], kind="mergesort")
        phi = df["phi"].to_numpy().reshape(ny, nx)

        profile_x = phi.mean(axis=0)
        fft_vals = np.fft.rfft(profile_x)

        this_kmax = min(k_max, fft_vals.size - 1)
        amps = np.zeros(k_max, dtype=float)
        if this_kmax > 0:
            amps[:this_kmax] = np.abs(fft_vals[1:this_kmax + 1]) / profile_x.size

        steps.append(int(df["step"].iloc[0]))
        mode_amp.append(amps)

    return np.asarray(steps), np.asarray(mode_amp)


results = {}
for label, folder in DATASETS:
    steps, amps = compute_modes(folder, K_MAX)
    t = steps if DT is None else steps * float(DT)
    results[label] = {"steps": steps, "amps": amps, "t": t}

t_label = "step" if DT is None else "time"

fig, axes = plt.subplots(5, 3, figsize=(15, 16), sharex=True)
axes = axes.ravel()

for k in range(1, K_MAX + 1):
    ax = axes[k - 1]
    for label, _folder in DATASETS:
        y = np.log10(np.maximum(results[label]["amps"][:, k - 1], EPS))
        ax.plot(results[label]["t"], y, lw=1.6, label=label)
    ax.set_title(f"k={k}")
    ax.grid(True, alpha=0.3)

for ax in axes[-3:]:
    ax.set_xlabel(t_label)
for i in [0, 3, 6, 9, 12]:
    axes[i].set_ylabel(r"$\log_{10}|\phi_k|$")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=len(DATASETS), frameon=False)
fig.suptitle("Mode comparison (phi)", y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

for label, _folder in DATASETS:
    steps = results[label]["steps"]
    print(f"{label}: {len(steps)} files, step range [{steps.min()}, {steps.max()}]")

In [ ]:
# Compare FFT mode amplitudes for diff_0 vs diff_10.
# Creates 15 subplots (k=1..15), each with both datasets overlaid.

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASETS = [
    ("diff_0", Path("diff_0")),
    ("diff_10", Path("diff_10")),
    # ("diff_100", Path("diff_100")),
    # ("diff_1000", Path("diff_1000")),
]
PATTERN = "E_field_samples_*x*_step_*.csv"
K_MAX = 15
COMPONENT = "Ex"  # "Ex" or "Ey"
DT = None  # set to simulation dt to plot physical time
EPS = 1e-30


def extract_step(path: Path):
    m = re.search(r"step_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else -1


def compute_modes(folder: Path, component: str, k_max: int):
    files = sorted(folder.glob(PATTERN), key=extract_step)
    if not files:
        raise FileNotFoundError(f"No files in {folder} matching {PATTERN}")

    steps = []
    mode_amp = []

    for f in files:
        df = pd.read_csv(f, usecols=["i", "j", "step", "Ex", "Ey"])
        if component not in df.columns:
            raise ValueError(f"Component {component} missing in {f.name}")

        nx = int(df["i"].max()) + 1
        ny = int(df["j"].max()) + 1
        if len(df) != nx * ny:
            raise ValueError(f"{f.name}: expected {nx*ny} rows, got {len(df)}")

        df = df.sort_values(["j", "i"], kind="mergesort")
        field = df[component].to_numpy().reshape(ny, nx)

        profile_x = field.mean(axis=0)
        fft_vals = np.fft.rfft(profile_x)

        this_kmax = min(k_max, fft_vals.size - 1)
        amps = np.zeros(k_max, dtype=float)
        if this_kmax > 0:
            amps[:this_kmax] = np.abs(fft_vals[1:this_kmax + 1]) / profile_x.size

        steps.append(int(df["step"].iloc[0]))
        mode_amp.append(amps)

    return np.asarray(steps), np.asarray(mode_amp)


results = {}
for label, folder in DATASETS:
    steps, amps = compute_modes(folder, COMPONENT, K_MAX)
    t = steps if DT is None else steps * float(DT)
    results[label] = {"steps": steps, "amps": amps, "t": t}

t_label = "step" if DT is None else "time"

fig, axes = plt.subplots(5, 3, figsize=(15, 16), sharex=True)
axes = axes.ravel()

for k in range(1, K_MAX + 1):
    ax = axes[k - 1]
    for label, _folder in DATASETS:
        y = np.log10(np.maximum(results[label]["amps"][:, k - 1], EPS))
        ax.plot(results[label]["t"], y, lw=1.6, label=label)
    ax.set_title(f"k={k}")
    ax.grid(True, alpha=0.3)

for ax in axes[-3:]:
    ax.set_xlabel(t_label)
for i in [0, 3, 6, 9, 12]:
    axes[i].set_ylabel(r"$\log_{10}|E_k|$")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=len(DATASETS), frameon=False)
fig.suptitle(f"Mode comparison ({COMPONENT})", y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

for label, _folder in DATASETS:
    steps = results[label]["steps"]
    print(f"{label}: {len(steps)} files, step range [{steps.min()}, {steps.max()}]")